In [1]:
#import necessary libararies
#process pdfs
from pypdf import PdfReader
from pathlib import Path
import glob
import numpy as np
import os

#create embeddings
from sentence_transformers import SentenceTransformer

#load embeddings
import chromadb

#language model
import ollama

In [2]:
folder_path = r"C:\Users\Sumed\Desktop\rag_assistant\data\*pdf"
pdf_files = glob.glob(folder_path)

if not pdf_files:
        raise Exception('No document(s) found')
else:
    print(f'{len(pdf_files)} document(s) found')

for i in pdf_files:
    print(i)

2 document(s) found
C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf


In [3]:
#some insights about the PDFs in the folder

page_counter = 0
total_words = 0
words_per_page = []
average_words_per_page = 0

for i in pdf_files:
    print('processing file : ', i)
    reader = PdfReader(i)
    max_words_per_page = 0

    for _,j in enumerate(reader.pages):
        current_page = j.extract_text()
        total_words_in_page = current_page.split()
        
        if max_words_per_page < len(total_words_in_page):
            max_words_per_page = len(total_words_in_page)
        words_per_page.append(len(total_words_in_page))

    print('max words in a page :',max_words_per_page) 
    print('total words in the PDF :',sum(words_per_page))
    print('average number of words per page: ', round(np.divide( sum(words_per_page),len(words_per_page) ),0) )
    print('\n')

processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
max words in a page : 1683
total words in the PDF : 49540
average number of words per page:  266.0


processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf
max words in a page : 596
total words in the PDF : 95328
average number of words per page:  298.0




In [4]:
# create chunks
all_chunks = []
chunks = {}
for i in pdf_files:
    print('processing file : ', i)
    reader = PdfReader(i)
    file_name = os.path.basename(i)
    page_counter = 0

    for _,j in enumerate(reader.pages):
        current_page = j.extract_text()
        chunks = {'source_file': file_name
                 ,'page_number': page_counter
                 ,'chunk_id'   : f'{file_name}_{page_counter}'
                 ,'text'       : current_page
        }
        page_counter+=1
        all_chunks.append(chunks)
    
print(f'sample chunks : {all_chunks[:5]}')
        

processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\encoder_decoder_neural_networks.pdf
processing file :  C:\Users\Sumed\Desktop\rag_assistant\data\Mathematics of Deep Learning.pdf
sample chunks : [{'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 0, 'chunk_id': 'encoder_decoder_neural_networks.pdf_0', 'text': 'ENCODER-DECODER\nNEURAL NETWORKS\nNal Kalchbrenner'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 1, 'chunk_id': 'encoder_decoder_neural_networks.pdf_1', 'text': 'Encoder-Decoder Neural Networks\nNal Kalchbrenner\nNew College\nUniversity of Oxford\nA thesis submitted for the degree of\nDoctor of Philosophy\nMichaelmas 2017'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 2, 'chunk_id': 'encoder_decoder_neural_networks.pdf_2', 'text': 'To my parents Metka and Bojan'}, {'source_file': 'encoder_decoder_neural_networks.pdf', 'page_number': 3, 'chunk_id': 'encoder_decoder_neural_networks.pdf_3', 'text': '

In [8]:
#create embeddings
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
text_to_embed = [i['text'] for i in all_chunks]
embeddings = embedding_model.encode(text_to_embed)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
#initialize chromadb with the persistent type so our vectors dont disappear
client = chromadb.PersistentClient(path=r'C:\Users\Sumed\Desktop\rag_assistant\chromadb_vectors\vectors')
collection = client.get_or_create_collection(name = 'vector_storage')

ids            = [i['chunk_id'] for i in all_chunks]
documents      = [i['text'] for i in all_chunks]
embedding_list = embeddings.tolist()
metadata       = [{'source_file': i['source_file'], 'page_number': i['page_number']} for i in all_chunks]

collection.add(ids       = ids
              ,documents = documents
              ,metadatas = metadata 
              ,embeddings = embedding_list  
           )

In [19]:
user_query = 'what is backpropagation?'
query_embeddings = embedding_model.encode(user_query)
# print(query_embeddings)

query_result = collection.query(query_embeddings=query_embeddings,n_results=5)

print(query_result)


{'ids': [['Mathematics of Deep Learning.pdf_91', 'Mathematics of Deep Learning.pdf_81', 'Mathematics of Deep Learning.pdf_90', 'Mathematics of Deep Learning.pdf_94', 'Mathematics of Deep Learning.pdf_74']], 'embeddings': None, 'documents': [['84 � 9 Backpropagation\nLet T ⊂ ℝn0 be a training set for the DNN (or a batch of the training set used in SGD), and\nlet L : ℝμ → ℝ be a loss function of the form (9.58). We now describe how to implement\nthe backpropagation algorithm to calculate ∇L(α) for a given α ∈ ℝμ:\n∇L(α) = ( 𝜕L\n𝜕α1\n, . . . , 𝜕L\n𝜕αM\n), (9.77)\nwhere each 𝜕L/𝜕αk is a vector corresponding to the sum of 𝜕ℓ/𝜕αk(s, .) for each object s\nin the batch. In turn, each 𝜕ℓ/𝜕αk(s, .) has μk components (see Fig. 9.6) given by\n𝜕L\n𝜕αk⏟⏟ ⏟⏟⏟ ⏟⏟\n1 × μk matrix\n= 𝜕ℓ\n𝜕hk⏟⏟ ⏟⏟⏟ ⏟⏟\n1 × nk matrix\n𝜕hk\n𝜕αk⏟⏟ ⏟⏟⏟ ⏟⏟\nnk × μk matrix\n. (9.78)\nFigure 9.6: Dimensions of matrices in (9.80).\n0. Forward propagation. Forward propagation (also referred to as a forward pass) is\nnot technicall

In [11]:
#initialize ollama and qwen3 with 4 billion parameters
response = ollama.chat(
    model="qwen3:4b",
    messages=[
        {
            "role": "user",
            "content": "how many letters are there in the word 'sumedh'?"
        }
    ]
    ,think = False
)
print(response)

model='qwen3:4b' created_at='2026-06-29T15:56:55.3070623Z' done=True done_reason='stop' total_duration=98623050400 load_duration=55693361300 prompt_eval_count=23 prompt_eval_duration=23998151000 eval_count=1314 eval_duration=18366232000 message=Message(role='assistant', content='First, the question is: "how many letters are there in the word \'sumedh\'?"\n\nI need to count the number of letters in the word "sumedh". But "sumedh" doesn\'t look like a standard English word. I think it might be a misspelling or a specific term. Let me think.\n\nIn English, common words like "sum" or "med" aren\'t typically combined that way. Perhaps it\'s meant to be "Sumedh" as a name. I recall that "Sumedh" might be a name from Indian context, like a surname or a place.\n\nBut the question is straightforward: it says "the word \'sumedh\'", so I should just count the characters as given.\n\nLet me write down the word: s-u-m-e-d-h.\n\nSo, the letters are: s, u, m, e, d, h.\n\nThat\'s six letters.\n\nI sho